# Case39 reinforcement planning from control completion

This notebook is the planning extension of the control-completion experiment.
The mathematical completion
\[
\Delta G_\lambda = Q/\lambda-G
\]
was introduced as a **virtual** relaxation for an existing grid.  Here we ask when
positive pieces of that missing control Gramian can instead be interpreted as
physical fast flexibility: battery storage, inverter-based resources, controllable
charging, flexible demand, or future grid reinforcement.

The central distinction is between **operational cost** and **planning cost**.
The SOC control metric \(R\) prices real-time actuation inside the stochastic-control
objective; it is not itself a capital-cost model.  We therefore report several
planning proxies separately: completion trace, effective MW-authority proxy, number
of installed devices, and scenario robustness.

The effective MW proxy used below follows the normalization
\[
P_i^{\rm eff}=S_{\rm base}\sqrt{r_{\rm ref}\,\Delta G_{ii}}.
\]
It is useful for comparisons but is not a complete battery nameplate model because
power saturation, state of charge, energy capacity and degradation are not included.

In [ ]:
import json, pandas as pd
from pathlib import Path
ROOT=Path.cwd().resolve().parents[0] if Path.cwd().name=='notebooks' else Path.cwd().resolve()
if ROOT.name!='case39': ROOT=Path('/mnt/data/SuboptimalControl_ReinforcementPlanning_09_09_26/experiments/case39')
d=json.load(open(ROOT/'results'/'reinforcement_planning.json'))
print(d['metadata'])

## 1. Exact-PIC reinforcement frontier

For the H3 generator architecture, exact scalar PIC compatibility is obtained by
physically adding enough control authority that
\[
G_{\rm reinforced}=Q/\lambda.
\]
The maximal admissible \(\lambda_{\max}\) is the **least** reinforcement and gives
the largest completed lower value.  Decreasing \(\lambda\) adds more authority and
lowers the operational value.  Thus \(\lambda\) parametrizes an exactly computable
investment-performance family inside the PIC-compatible design class.

Aggressive small-\(\lambda\) points have lower importance-sampling ESS; they are
included to show the trend and should not be read as finite-sample rigorous bounds.

In [ ]:
for kind in ['homogeneous','activity_scaled']:
    print('\n',kind)
    display(pd.DataFrame(d['frontier'][kind])[[ 'lambda_fraction','completion_trace','effective_power_authority_MW','W','ess_fraction']])

## 2. Designer freedom in the control metric

For independent generator controls, let \(g_i=1/r_i\) be the physical control
authority.  Scalar PIC matching requires
\[
\lambda g_i\le \Gamma_i^2.
\]
With heterogeneous noise a fixed common \(R=rI\) can create a weakest-channel
bottleneck.  Holding total generator authority \(\sum_i g_i\) fixed, choosing
\(g_i\propto \Gamma_i^2\) makes all controlled channels hit the matching condition
at the same \(\lambda\).  This does not change the homogeneous case but sharply
reduces completion burden for activity-scaled noise.

In [ ]:
for kind in ['homogeneous','activity_scaled']:
    print('\n',kind)
    for name,r in d['metric_design'][kind].items():
        print(name,'lambda=',r['lambda_value'],'trDelta=',r['completion_trace'],'W+=',r['W'],'J=',r['policy']['J_mean'])

## 3. Fixed versus scenario-reconfigurable flexibility

A fixed reinforced grid generally cannot be exactly scalar-PIC matched to two
non-proportional covariance matrices.  We therefore compare:

* **uniform** generator authority;
* an exact minimax **robust fixed** allocation that minimizes the worst completion
  burden over the homogeneous and activity-scaled scenarios under the same total
  authority budget;
* an **activity-specific** allocation, representing reconfigurable or mobile
  flexibility positioned for the forecast scenario.

For scenario \(s\), if \(q_s=\Gamma_s^2\), the fixed-authority burden obeys
\[
B_s(g)+A=(\sum_i q_{s,i})\max_{g\in\mathcal G}\frac{g_i}{q_{s,i}},
\qquad A=\sum_i g_i.
\]
The two-scenario minimax problem has a closed-form epigraph solution.

In [ ]:
for kind in ['homogeneous','activity_scaled']:
    print('\n',kind)
    rr=d['scenario_flexibility'][kind]
    for n in ['uniform','robust_fixed','activity_specific']:
        r=rr[n];print(n,'trDelta=',r['completion_trace'],'W+=',r['W'],'J=',r['policy']['J_mean'])
print('robust burdens:',d['scenario_flexibility']['robust_optimizer']['burdens'])

## 4. Sparse physical reinforcement

Exact completion may require many devices.  To test whether the virtual oracle can
still guide a sparse investment, we add only \(k\) load-bus actuators on top of H3
and synthesize a local LQR feedback for the reinforced nonlinear swing simulation.
Two placement heuristics are compared:

1. **noise ranking:** largest unactuated \(\Gamma_i^2\);
2. **dynamic shadow proxy:** largest RMS nodal command from an H4 completed LQR
   guide along the severe-event stochastic trajectory.

The second is only a proxy for the exact value-function shadow price
\(\tfrac12\mathbb E\int(a^T\nabla J)^2dt\); computing that exact sensitivity would
require the gradient of the completed PIC value.

In [ ]:
for kind in ['homogeneous','activity_scaled']:
    print('\n',kind)
    for method in ['noise','shadow_proxy']:
        print(method);display(pd.DataFrame(d['sparse'][kind][method]))

## 5. Current interpretation

The case39 tests support **completion-aware reinforcement**, not the stronger claim
that a future grid should be designed to be exactly PIC.  Exact PIC completion has
three advantages: a solvable nonlinear stochastic-control value, an automatic lower
certificate, and a transparent map from noise geometry to missing control authority.
But exact matching can be capital intensive and a sparse, non-PIC reinforcement can
still reduce the deployed nonlinear cost efficiently.  The proper engineering object
is therefore the *price of linearisability*: compare the best PIC-compatible design
against the best unrestricted reinforcement under the same capital and safety
constraints.  This notebook establishes the ingredients for that comparison, not a
final economic optimum.

In [ ]:
from IPython.display import Image,display
for f in ['case39_pic_planning_frontier.png','case39_control_metric_design.png','case39_scenario_flexibility.png','case39_sparse_reinforcement.png']:
    display(Image(filename=str(ROOT/'figures'/f)))